In [ ]:
#@title Cell 47.1 — Purpose and interpretation
# This notebook asks whether genomic-similarity groups improve *antibiotic-specific*
# linear calibration of Model C predictions. It is a descriptive/exploratory
# comparison, not a chromosome-wide SNP or chromosomal MIC score analysis.
#
# Three held-out point predictors per antibiotic:
#   A: original Model C prediction r
#   B: global training-only regression a_antibiotic + beta_antibiotic * r
#   C: genomic-group-specific training-only regression a_group + beta_group * r
#
# The four genomic groups are derived *without MIC values* from Model C's
# saved pathogen embeddings, and only outer-TRAIN pathogens are used to
# define their KMeans centres. Outer-TEST genomes are assigned to a centre.
# Genomic representations are from Model C (including AMR/sequence information);
# the groups are NOT chromosome-only clusters or phylogenetic lineages.
# The saved embeddings themselves were constructed for the full genome cohort,
# so this is a genome-transductive, phenotype-held-out analysis.
#
# Predictions and fit parameters use the original five pathogen-out folds.
# No outer-TEST observed MIC is used for clustering or fitting slopes.
# No model is selected based on these held-out results; follow-up confirmation
# would be required before adopting a genomic-group correction.

print('NB47: Antibiotic versus genomic-similarity-group calibration')
print('Original 50,460 held-out observations; 9,058 BioSamples; 26 antibiotics.')


In [ ]:
#@title Cell 47.2 — Find original inputs and Model C embeddings automatically
import os, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC project folder not found: {ROOT}')
SEARCH=Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
OUT=ROOT/'03_Results'/'47_ModelC_Genomic_Group_Slope_Audit'
OUT.mkdir(parents=True,exist_ok=True)

CALIB_NAME='40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'
TEST_NAME='40_BMC_independent_outer_test_predictions.csv.gz'
INTER_NAME='26_model_c_aligned_interactions.csv.gz'
EMB_NAME={
 'locus_rho_0p3':'26_locus_rho_0p3_embedding.npz',
 'locus_rho_0p4':'26_locus_rho_0p4_embedding.npz',
}

def search_exact(names):
    targets=set(names)
    hits={n:[] for n in targets}
    for folder,dirs,files in os.walk(SEARCH):
        dirs[:]=[d for d in dirs if d not in {'.Trashes','.Trash-1000','.git','__pycache__','node_modules'}]
        for fn in targets.intersection(files):hits[fn].append(Path(folder)/fn)
    return hits

all_names=[CALIB_NAME,TEST_NAME,INTER_NAME,*EMB_NAME.values()]
found=search_exact(all_names)
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def prefer(name):
    candidates=found[name]
    if not candidates:
        raise FileNotFoundError('Required file not found anywhere under '+str(SEARCH)+': '+name)
    def priority(p):
        # Prefer original RefMIC project, then NB40 reconstructed embeddings,
        # then original Model C checkpoints in another Drive directory.
        return (0 if ROOT in p.parents else 1,0 if '40_BMC_Training_Only_Calibration' in str(p) else 1,
                len(p.parts),str(p))
    candidates=sorted(candidates,key=priority)
    if len(candidates)>1:
        hashes={digest(p) for p in candidates}
        if len(hashes)>1:
            # Distinct historical copies can differ; select an NB40 verified
            # embedding only when the notebook explicitly marked it verified.
            marked=[p for p in candidates if p.name.endswith('_embedding.npz')
                    and p.with_suffix('.verified.json').is_file()
                    and json.loads(p.with_suffix('.verified.json').read_text()).get('sha256')==digest(p)
                    and json.loads(p.with_suffix('.verified.json').read_text()).get('validation_status')=='passed']
            if len(marked)==1:
                print('Using NB40-verified embedding among differing copies:',marked[0])
                return marked[0]
            raise ValueError('Different copies of '+name+' were found. Please retain one authoritative copy:\n'+
                             '\n'.join(map(str,candidates)))
        print(f'{name}: found {len(candidates)} identical copies; using {candidates[0]}')
    return candidates[0]

CALIB_PATH=prefer(CALIB_NAME)
TEST_PATH=prefer(TEST_NAME)
INTER_PATH=prefer(INTER_NAME)
EMBED_PATH={cid:prefer(name) for cid,name in EMB_NAME.items()}
for key,p in [('Training-only inner predictions',CALIB_PATH),('Independent outer predictions',TEST_PATH),
              ('Original interaction index',INTER_PATH),*[(cid,p) for cid,p in EMBED_PATH.items()]]:
    print(f'{key}: {p}')
print('Results folder:',OUT)


In [ ]:
#@title Cell 47.3 — Load inputs and check the five held-out folds
calib=pd.read_csv(CALIB_PATH,compression='gzip')
test=pd.read_csv(TEST_PATH,compression='gzip')
inter=pd.read_csv(INTER_PATH,compression='gzip',usecols=['biosample','model_c_pathogen_row'])
need_calib={'outer_fold_for_calibration','interaction_row','biosample','antibiotic',
            'observed_log2_mic','inner_oof_predicted_log2_mic','candidate_id'}
need_test={'outer_fold','interaction_row','biosample','antibiotic','observed_log2_mic',
           'model_c_outer_test_predicted_log2_mic'}
for name,df,required in [('calibration',calib,need_calib),('held-out',test,need_test)]:
    absent=required-set(df.columns)
    if absent:raise ValueError(f'{name}: missing required columns: {sorted(absent)}')
if (len(calib),len(test),test.biosample.nunique(),test.antibiotic.nunique())!=(201840,50460,9058,26):
    raise ValueError('Input dataset dimensions differ from the validated NB40 outputs.')
if test.duplicated('interaction_row').any() or test.duplicated(['biosample','antibiotic']).any():
    raise ValueError('Held-out observations are duplicated')
if set(test.outer_fold.unique())!=set(range(1,6)):
    raise ValueError('Five original outer folds are required')
if test.groupby('biosample').outer_fold.nunique().max()!=1:
    raise ValueError('One pathogen appears in multiple outer test folds')

pmap=inter.drop_duplicates('biosample').copy()
if len(pmap)!=9058 or pmap.model_c_pathogen_row.nunique()!=9058:
    raise ValueError('Incomplete or nonunique genomic embedding index')
pmap['biosample']=pmap.biosample.astype(str)
idx_map=pmap.set_index('biosample').model_c_pathogen_row.astype(int).to_dict()
if set(test.biosample.astype(str))!=set(idx_map):
    raise ValueError('Genomic embedding index is not identical to held-out BioSamples')
if not set(calib.biosample.astype(str)).issubset(idx_map):
    raise ValueError('Calibration BioSample missing from genomic embedding index')

settings=calib.groupby('outer_fold_for_calibration',sort=True).candidate_id.agg(['first','nunique'])
if (settings['nunique']!=1).any() or not set(settings['first']).issubset(EMBED_PATH):
    raise ValueError('Unexpected candidate embeddings in calibration records')
EMBED={}
for cid,path in EMBED_PATH.items():
    with np.load(path,allow_pickle=False) as z:
        if 'pathogen_embedding' not in z:
            raise ValueError(f'Missing pathogen_embedding in {path}')
        v=np.asarray(z['pathogen_embedding'],dtype=np.float32)
    if v.shape!=(9058,256) or not np.isfinite(v).all():
        raise ValueError(f'{path.name}: expected 9058 × 256 finite embeddings, got {v.shape}')
    EMBED[cid]=v

fold_audit=[]
for fold in range(1,6):
    c=calib[calib.outer_fold_for_calibration.eq(fold)]
    t=test[test.outer_fold.eq(fold)]
    if set(c.biosample)&set(t.biosample):
        raise ValueError(f'Outer test pathogen leakage in fold {fold}')
    if len(c)!=len(test)-len(t):
        raise ValueError(f'Fold {fold}: missing calibration rows')
    if set(c.interaction_row)!=set(test.interaction_row)-set(t.interaction_row):
        raise ValueError(f'Fold {fold}: calibration interaction mismatch')
    fold_audit.append({'outer_fold':fold,'n_train_pathogens':c.biosample.nunique(),
                       'n_test_pathogens':t.biosample.nunique(),
                       'candidate_id':settings.loc[fold,'first'],
                       'no_test_pathogens_in_training':True})
print('Input identity and pathogen-out leakage checks passed.')
display(pd.DataFrame(fold_audit))


In [ ]:
#@title Cell 47.4 — Create four genomic-similarity groups per outer fold (no MIC input)
# Fixed exploratory design, not tuned using held-out MICs.
K_GROUPS=4
RANDOM_SEED=47
MIN_GROUP_TRAIN_N=40
MIN_GROUP_PRED_SD=0.15

assignments=[]
cluster_info=[]
for fold in range(1,6):
    cid=settings.loc[fold,'first'];e=EMBED[cid]
    c=calib[calib.outer_fold_for_calibration.eq(fold)]
    t=test[test.outer_fold.eq(fold)]
    train_ids=np.sort(c.biosample.astype(str).unique())
    test_ids=np.sort(t.biosample.astype(str).unique())
    xtrain=np.asarray(e[[idx_map[bs] for bs in train_ids]],dtype=np.float32)
    xtest=np.asarray(e[[idx_map[bs] for bs in test_ids]],dtype=np.float32)
    if set(train_ids)&set(test_ids):raise AssertionError('Train/test pathogen overlap')
    # The grouping depends only on genomic embeddings, not observed MIC values.
    km=KMeans(n_clusters=K_GROUPS,random_state=RANDOM_SEED,n_init=10,max_iter=300)
    train_groups=km.fit_predict(xtrain).astype(int)
    test_groups=km.predict(xtest).astype(int)
    for partition,ids,groups in [('train',train_ids,train_groups),('test',test_ids,test_groups)]:
        assignments.append(pd.DataFrame({'outer_fold':fold,'biosample':ids,
                                         'partition':partition,'genomic_group':groups}))
        for g,n in pd.Series(groups).value_counts().sort_index().items():
            cluster_info.append({'outer_fold':fold,'candidate_id':cid,
                                 'partition':partition,'genomic_group':int(g),
                                 'n_pathogens':int(n)})
assignment=pd.concat(assignments,ignore_index=True)
cluster_summary=pd.DataFrame(cluster_info)
if assignment.duplicated(['outer_fold','biosample']).any():
    raise ValueError('Duplicate group assignments per fold')
if len(assignment)!=5*9058:
    raise ValueError('Not all pathogens assigned in every outer fold')
if not (cluster_summary[cluster_summary.partition.eq('train')].groupby('outer_fold').genomic_group.nunique()==K_GROUPS).all():
    raise ValueError('Some outer training partition contains fewer than four groups')
print('Training-defined groups are established without MIC values.')
print('Group labels are arbitrary; group 0 in fold 1 need not match group 0 in fold 2.')
display(cluster_summary.pivot_table(index=['outer_fold','genomic_group'],columns='partition',values='n_pathogens').reset_index())


In [ ]:
#@title Cell 47.5 — Fit antibiotic-only and group-specific slopes on training predictions
def ordinary_linear(t):
    x=t.inner_oof_predicted_log2_mic.to_numpy(dtype=np.float64)
    y=t.observed_log2_mic.to_numpy(dtype=np.float64)
    avgx=x.mean();avgy=y.mean()
    denom=float(np.sum((x-avgx)**2))
    if denom<1e-12:raise ValueError('Insufficient variation in training predictions')
    beta=float(np.dot(x-avgx,y-avgy)/denom)
    a=float(avgy-beta*avgx)
    return a,beta,float(x.std())

train_groups=assignment[assignment.partition.eq('train')][['outer_fold','biosample','genomic_group']]
train=calib.merge(train_groups,left_on=['outer_fold_for_calibration','biosample'],
                  right_on=['outer_fold','biosample'],how='left',validate='many_to_one')
if train.genomic_group.isna().any():raise ValueError('Unassigned training observation')

fitted=[]
for (fold,drug),part in train.groupby(['outer_fold_for_calibration','antibiotic'],sort=True):
    a_global,b_global,s_global=ordinary_linear(part)
    if len(part)<40:raise ValueError(f'Not enough calibration rows for {fold}, {drug}')
    for group in range(K_GROUPS):
        sub=part[part.genomic_group.eq(group)]
        n=len(sub)
        if n>=MIN_GROUP_TRAIN_N and sub.inner_oof_predicted_log2_mic.std(ddof=0)>=MIN_GROUP_PRED_SD:
            a,b,xstd=ordinary_linear(sub)
            source='group_OLS'
        else:
            a,b,xstd=a_global,b_global,float(sub.inner_oof_predicted_log2_mic.std(ddof=0)) if n else 0.0
            source='antibiotic_fallback'
        fitted.append({'outer_fold':int(fold),'antibiotic':drug,'genomic_group':group,
                       'n_training':n,'n_train_genomic_groups':K_GROUPS,
                       'group_prediction_sd_log2':xstd,
                       'group_intercept_log2':a,'group_slope':b,
                       'antibiotic_intercept_log2':a_global,'antibiotic_slope':b_global,
                       'fit_source':source})
fits=pd.DataFrame(fitted)
if len(fits)!=5*26*K_GROUPS or fits.duplicated(['outer_fold','antibiotic','genomic_group']).any():
    raise ValueError('Expected exactly 520 fold-antibiotic-group fits')
print('Group-specific regressions:',len(fits))
print('Training groups using antibiotic-only fallback:',int(fits.fit_source.eq('antibiotic_fallback').sum()))
print('For comparison, the antibiotic-only fit uses all training pathogens in the same fold.')
display(fits[fits.antibiotic.isin(['ceftazidime','cefepime','ciprofloxacin','meropenem'])][
    ['outer_fold','antibiotic','genomic_group','n_training','antibiotic_slope',
     'group_slope','fit_source']].round(3).head(32))


In [ ]:
#@title Cell 47.6 — Evaluate three point predictors on the same independent pathogens
test_groups=assignment[assignment.partition.eq('test')][['outer_fold','biosample','genomic_group']]
paired=test.merge(test_groups,on=['outer_fold','biosample'],how='left',validate='many_to_one')
paired=paired.merge(fits[['outer_fold','antibiotic','genomic_group','group_intercept_log2',
                          'group_slope','antibiotic_intercept_log2','antibiotic_slope','fit_source']],
                    on=['outer_fold','antibiotic','genomic_group'],how='left',validate='many_to_one')
if len(paired)!=50460 or paired.genomic_group.isna().any() or paired.group_slope.isna().any():
    raise ValueError('Test observations missing a training-only correction')

r=paired.model_c_outer_test_predicted_log2_mic
paired['pred_model_c_log2']=r
paired['pred_antibiotic_linear_log2']=paired.antibiotic_intercept_log2+paired.antibiotic_slope*r
paired['pred_genomic_group_linear_log2']=paired.group_intercept_log2+paired.group_slope*r
for label in ['model_c','antibiotic_linear','genomic_group_linear']:
    paired[f'residual_{label}_log2']=paired.observed_log2_mic-paired[f'pred_{label}_log2']

def quantify(d,method):
    v=d[f'residual_{method}_log2'].to_numpy(dtype=np.float64)
    return {'MAE_log2':float(np.mean(np.abs(v))),'RMSE_log2':float(np.sqrt(np.mean(v*v))),
            'under_ge2':int((v>=2).sum()),'under_ge4':int((v>=4).sum()),
            'over_ge2':int((v<=-2).sum()),'largest_underprediction_log2':float(v.max())}

records=[]
for drug,df in [('ALL',paired)]+list(paired.groupby('antibiotic',sort=True)):
    for meth in ['model_c','antibiotic_linear','genomic_group_linear']:
        records.append({'antibiotic':drug,'method':meth,'n_test':len(df),'n_pathogens':df.biosample.nunique(),
                        **quantify(df,meth)})
metrics=pd.DataFrame(records)
all_rows=metrics[metrics.antibiotic.eq('ALL')].copy()
print('Same 50,460 independent held-out MIC observations:')
display(all_rows.round(4))

v=metrics[metrics.antibiotic.ne('ALL')].pivot(index='antibiotic',columns='method',values=['MAE_log2','RMSE_log2','under_ge2','under_ge4'])
summary=pd.DataFrame(index=v.index)
for m in ['MAE_log2','RMSE_log2','under_ge2','under_ge4']:
    for mode in ['model_c','antibiotic_linear','genomic_group_linear']:
        summary[f'{m}_{mode}']=v[(m,mode)]
    summary[f'{m}_gain_group_vs_antibiotic']=v[(m,'antibiotic_linear')]-v[(m,'genomic_group_linear')]
summary=summary.reset_index()
print('Group correction versus antibiotic-only linear correction (positive gain is better):')
display(summary[['antibiotic','MAE_log2_antibiotic_linear','MAE_log2_genomic_group_linear',
                 'MAE_log2_gain_group_vs_antibiotic',
                 'under_ge2_antibiotic_linear','under_ge2_genomic_group_linear',
                 'under_ge4_antibiotic_linear','under_ge4_genomic_group_linear']].sort_values('MAE_log2_gain_group_vs_antibiotic',ascending=False).round(4))
print('This is exploratory: do not choose the final model based on the held-out table alone.')


In [ ]:
#@title Cell 47.7 — Plot antibiotic-specific versus genomic-group-specific slopes
FOCUS=['ceftazidime','cefepime','ciprofloxacin','meropenem']
fig,axs=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
for ax,drug in zip(axs.flat,FOCUS):
    z=fits[fits.antibiotic.eq(drug)]
    for fold in range(1,6):
        q=z[z.outer_fold.eq(fold)]
        for _,p in q.iterrows():
            x=fold+(p.genomic_group-1.5)*0.12
            ax.scatter(x,p.group_slope,s=30,alpha=0.85,marker='o' if p.fit_source=='group_OLS' else 'x',
                       color=f'C{int(p.genomic_group)}')
        ax.scatter(fold,q.antibiotic_slope.iloc[0],s=80,color='black',marker='_',linewidths=2)
    ax.axhline(1,ls='--',lw=1,color='gray')
    ax.set_title(drug)
    ax.set_xlabel('Original outer fold')
    ax.set_ylabel('Training-only fitted slope')
    ax.set_xticks(range(1,6))
    ax.grid(axis='y',alpha=.2)
fig.suptitle('Four focus antibiotics: group-specific slopes (colours), antibiotic-only slope (black dash)',fontsize=12)
f=OUT/'47_Figure1_Group_Slopes_Focus_Antibiotics.png';fig.savefig(f,dpi=180,bbox_inches='tight')
plt.show();print('Saved:',f)


In [ ]:
#@title Cell 47.8 — Plot MAE and upper-tail underprediction differences across 26 antibiotics
summary['under_ge2_gain_group_vs_antibiotic']=summary.under_ge2_antibiotic_linear-summary.under_ge2_genomic_group_linear
z=summary.sort_values('MAE_log2_gain_group_vs_antibiotic')
fig,axs=plt.subplots(1,2,figsize=(15,9),sharey=True,constrained_layout=True)
yloc=np.arange(len(z))
for ax,metric,title in [
    (axs[0],'MAE_log2_gain_group_vs_antibiotic','MAE gain: group-specific minus antibiotic-only calibration'),
    (axs[1],'under_ge2_gain_group_vs_antibiotic','Reduction in ≥2-log₂ underpredictions')]:
    ax.barh(yloc,z[metric].astype(float),color='gray')
    ax.axvline(0,color='black',lw=0.9)
    ax.set_yticks(yloc,labels=z.antibiotic,fontsize=9)
    ax.set_title(title,fontsize=11)
    ax.grid(axis='x',alpha=.2)
axs[0].set_xlabel('Positive = group-specific better (log₂ MIC)')
axs[1].set_xlabel('Positive = fewer underpredictions (observations)')
fig.suptitle('Independent outer-test comparison, fixed four genomic groups per training fold',fontsize=13)
f=OUT/'47_Figure2_26_Antibiotic_Group_Correction_Comparison.png';fig.savefig(f,dpi=180,bbox_inches='tight')
plt.show();print('Saved:',f)


In [ ]:
#@title Cell 47.9 — Visualise 25 largest original underpredictions and save all outputs
# Same original severe cases for all methods; selected only for visualisation
# AFTER independent testing, not used for choosing slopes or genome groups.
for drug in FOCUS:
    q=paired[paired.antibiotic.eq(drug)].nlargest(25,'residual_model_c_log2').copy()
    q=q.sort_values('residual_model_c_log2',ascending=True)
    yy=np.arange(len(q))
    fig,ax=plt.subplots(figsize=(12,9))
    for mode,color,marker,lab,offset in [
        ('model_c','gray','o','Model C',-0.18),
        ('antibiotic_linear','tab:blue','s','Antibiotic-only linear',0),
        ('genomic_group_linear','tab:orange','D','Genomic-group linear',0.18)]:
        ax.scatter(q[f'pred_{mode}_log2'],yy+offset,s=30,marker=marker,color=color,label=lab,alpha=.9)
    ax.scatter(q.observed_log2_mic,yy,s=75,color='black',marker='x',label='Observed log₂(MIC)')
    ax.set_yticks(yy,q.biosample,fontsize=8)
    ax.set_xlabel('log₂(MIC)')
    ax.set_title(f'{drug}: 25 largest original Model C underpredictions (same held-out pathogens)')
    ax.grid(axis='x',alpha=.2)
    ax.legend(loc='lower right',fontsize=9)
    f=OUT/f'47_Figure3_25_Original_Underpredictions_{drug}.png';fig.savefig(f,dpi=180,bbox_inches='tight')
    plt.show();print('Saved:',f)

# Save everything; never overwrite NB40/NB45 results.
for filename,df in [
    ('47_fold_genomic_group_assignments.csv.gz',assignment),
    ('47_group_sizes_by_fold.csv',cluster_summary),
    ('47_training_only_antibiotic_and_group_slopes.csv',fits),
    ('47_three_method_held_out_point_predictions.csv.gz',paired),
    ('47_three_method_overall_and_antibiotic_metrics.csv',metrics),
    ('47_26_antibiotic_group_correction_comparison.csv',summary),
]:
    p=OUT/filename
    df.to_csv(p,index=False,compression='gzip' if p.suffix=='.gz' else None)
    print('Saved:',p)
report={
 'n_outer_folds':5,'n_biosamples':int(test.biosample.nunique()),
 'n_observations':len(test),'n_antibiotics':int(test.antibiotic.nunique()),
 'n_training_calibration_rows_5_overlapping_outer_training_sets':len(calib),
 'n_genomic_groups_per_fold':K_GROUPS,
 'n_fits_fold_antibiotic_group':len(fits),
 'n_fallback_groups':int(fits.fit_source.eq('antibiotic_fallback').sum()),
 'grouping_used_observed_MIC':False,'slopes_used_outer_heldout_MIC':False,
 'embedding_scope':'full genome cohort, features generated without the MIC outcome',
 'conclusion_scope':'exploratory; not proof of genotype-specific slope or validated final model',
}
(OUT/'47_validation_audit.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('NB47 complete. Please share Cell 47.6 overall table and Figure 1 group slopes.')
